## Q&A Losses

**Work in progress**: Aggregate DM-Mathematics token losses into question and answers losses

In [2]:
import os
import os
import numpy as np
import pickle
import pandas as pd

In [4]:
# Requires some local data
# https://drive.google.com/drive/u/0/folders/1KAjqTusn0e9EVe6t1nwbNzs62TFUZx1M
data_folder = "~/code/extract_psych/pile_losses"  # your path


In [5]:
mapping = {'pythia_(70m)': 'eleutherai/pythia-70m',
 'falcon_(7b)': 'tiiuae/falcon-7b',
 'gemma_(7b)': 'google/gemma-7b',
 'gemma_2_instruct_(9b)': 'google/gemma-2-9b-it',
 'llama2_7b': 'meta-llama/llama-2-7b-hf',
 'llama2_13b': 'meta-llama/llama-2-13b-hf',
 'llama3_8b': 'meta-llama/meta-llama-3-8b',
 'llama3.2_1b': 'meta-llama/llama-3.2-1b-instruct',
 'llama3.2_3b': 'meta-llama/llama-3.2-3b',
 'llama3.2_3b_instruct': 'meta-llama/llama-3.2-3b-instruct',
 'mistral_instruct_v0.3_(7b)': 'mistralai/mistral-7b-instruct-v0.3',
 'mistral_nemo_(2402)': 'mistralai/mistral-nemo-base-2407',
 'mistral_v0.1_(7b)': 'mistralai/mistral-7b-v0.1',
 'phi-2': 'microsoft/phi-2',
 'phi-3_(7b)': 'microsoft/phi-3-small-8k-instruct',
 'phi-3_(14b)': 'microsoft/phi-3-medium-4k-instruct',
 'qwen1.5_(7b)': 'qwen/qwen1.5-7b',
 'qwen1.5_(14b)': 'qwen/qwen1.5-14b',
 'qwen2.5_instruct_(7b)': 'qwen/qwen2.5-7b-instruct',
 't0pp_(11b)': 'bigscience/t0pp',
 'biomistral_(7b)': 'biomistral/biomistral-7b',
 'dolly_v2_(3b)': 'databricks/dolly-v2-3b',
 'dolly_v2_(7b)': 'databricks/dolly-v2-7b',
 'dolly_v2_(12b)': 'databricks/dolly-v2-12b',
 'gpt-j_(6b)': 'eleutherai/gpt-j-6b',
 'pythia_(1b)': 'eleutherai/pythia-1b',
 'pythia_(2.8b)': 'eleutherai/pythia-2.8b',
 'pythia_(6.9b)': 'eleutherai/pythia-6.9b',
 'pythia_(12b)': 'eleutherai/pythia-12b',
 'meditron_(7b)': 'epfl-llm/meditron-7b',
 'gemma_(2b)': 'google/gemma-2b',
 'gemma_instruct_(2b)': 'google/gemma-2b-it',
 'gemma_instruct_(7b)': 'google/gemma-7b-it',
 'gemma_2_(2b)': 'google/gemma-2-2b',
 'gemma_2_(9b)': 'google/gemma-2-9b',
 'vicuna_v1.3_(7b)': 'lmsys/vicuna-7b-v1.3',
 'vicuna_v1.3_(13b)': 'lmsys/vicuna-13b-v1.3',
 'yi_(6b)': '01-ai/yi-6b',
 'yi_chat_(6b)': '01-ai/yi-6b-chat',
 'qwen1.5_chat_(7b)': 'qwen/qwen1.5-7b-chat',
 'qwen1.5_chat_(14b)': 'qwen/qwen1.5-14b-chat',
 'sailor_(7b)': 'sail/sailor-7b',
 'sailor_chat_(7b)': 'sail/sailor-7b-chat',
 'typhoon_(7b)': 'scb10x/typhoon-7b',
 'stablelm-base-alpha_(3b)': 'stabilityai/stablelm-base-alpha-3b',
 'stablelm-base-alpha_(7b)': 'stabilityai/stablelm-base-alpha-7b',
 'sabia_7b': 'maritaca-ai/sabia-7b',
 'granite_3.1_-_8b_-_base': 'ibm-granite/granite-3.1-8b-base',
 'granite_3.1_-_8b_-_instruct': 'ibm-granite/granite-3.1-8b-instruct',
 'granite_3.1_-_2b_-_instruct': 'ibm-granite/granite-3.1-2b-instruct',
 'granite_3.1_-_2b_-_base': 'ibm-granite/granite-3.1-2b-base',
 'deepseek-r1-distill-llama-8b': 'deepseek-ai/deepseek-r1-distill-llama-8b',
 'deepseek-coder-6.7b-instruct': 'deepseek-ai/deepseek-coder-6.7b-instruct',
 'pythia-14m': 'eleutherai/pythia-14m',
 'pythia-160m': 'eleutherai/pythia-160m',
 'pythia-31m': 'eleutherai/pythia-31m',
 'pythia-410m': 'eleutherai/pythia-410m',
 'llama-3.2-1b': 'meta-llama/llama-3.2-1b',
 'llama-3-typhoon-v1.5-8b': 'scb10x/llama-3-typhoon-v1.5-8b',
 'llama-3-typhoon-v1.5-8b-instruct': 'scb10x/llama-3-typhoon-v1.5-8b-instruct'};

In [6]:

# extend mapping to support both naming conventions
for v in list(mapping.values()):
    alt = v.split("/")[1]  # the extra models follow this convention
    mapping[alt] = v

data_folder = os.path.expanduser(data_folder)
files = [f for f in os.listdir(data_folder) if os.path.isfile(os.path.join(data_folder, f))]
models = [f[:-4].lower() for f in files if f.endswith(".pkl")]
missing = [m for m in models if m not in mapping]
models = [m for m in models if m in mapping]
print(f"Found {len(models)} viable models!")


Found 58 viable models!


In [8]:
# Lets confirm what the contexts are - will be useful for other things
model = models[0]
file_path = os.path.join(data_folder, model + ".pkl")
with open(file_path, 'rb') as f:
    model_dict = pickle.load(f)

In [ ]:
subsets = model_dict.keys() - ['dm_math_categories']
text = {}
for subset in subsets:
    contexts = model_dict[subset]
    for id, tokens in zip(contexts["context_id"], contexts["tokens"]):
        name = f"{subset} {id:03d}"
        text[name] = "".join(tokens)

pd.DataFrame.from_dict(text, orient='index').to_csv("contexts.csv")

In [ ]:
def split_QA(tokens, item_losses):
    # So here we might assume the "\n" is a standalone token - will this asumption hold?
    parts = []
    losses = []
    part = ""
    loss = 0.

    for token, token_loss in zip(tokens, item_losses):
        if token == "\n":
            # loss += token_loss  # Do we care about the newline loss here?
            parts.append(part)
            losses.append(loss)
            part = ""
            loss = 0.
        else:
            part += token
            loss += token_loss
    if len(part):
        # and the incomplete bit at the end
        parts.append(part)
        losses.append(loss)
    # Now guess which one is the answer
    count0 = sum(len(f) for f in parts[::2])
    count1 = sum(len(f) for f in parts[1::2])

    if count0 < count1:
        # starts with an answer with no question
        # That's clearly not an "answer" - ignore
        parts = parts[1:]
        losses = losses[1:]
    
    question_loss = float(np.mean(losses[::2]))
    answer_loss = float(np.mean(losses[1::2]))
    #print(parts[0])  # should always be a question - yes it is
    #print("|".join(parts[::2]))
    return question_loss, answer_loss

In [ ]:
def extract_answers(model_dict, aggregate=True):

    # Split pile_subsets_mini's dm_mathematics into question and answer losses
    cats = model_dict['dm_mathematics']
    cats = pd.DataFrame(cats).to_dict('records')
    rows = []
    cols = {}
    for item in cats:
        context_id = item['context_id']
        q_losses, a_losses = split_QA(item['tokens'], item['loss'])
        cols[f"Pile-Q{context_id:04d}"] = q_losses
        cols[f"Pile-A{context_id:04d}"] = a_losses

    colsM = {}
    for item in model_dict['dm_math_categories']:
        # if "context_idx" not in item:
        #     item["context_idx"] = item["context_id"]
        context_id = item['category'].replace(" ", "")
        if not aggregate:
            # if aggregate is true we just store the category
            context_id = f"{context_id}:{item['context_idx']:03d}"

        cutoff = item['tokens'].index("\n")
        zeroshot = item['loss']['zero-shot']
        fewshot = item['loss']['few-shot']

        # "".join(item['tokens'][:cutoff])
        # "".join(item['tokens'][cutoff+1:])
        extract = [
            (f"{context_id}-ZSQ", zeroshot[:cutoff]),
            (f"{context_id}-ZSA", zeroshot[cutoff+1:]),
            (f"{context_id}-FSQ", fewshot[:cutoff]),
            (f"{context_id}-FSA", fewshot[cutoff+1:]),
        ]
        
        for key, vals in extract:
            if key not in colsM:
                colsM[key] = []
            colsM[key].extend(vals)

    # Then aggregate - again do we use mean or sum aggregation? (it probably doesn't matter)
    colsa = {k: float(np.mean(v)) for k, v in colsM.items()}
    cols.update(colsa)
    return cols


In [ ]:
rows = []
for model in models:
    model_id = mapping[model]
    file_path = os.path.join(data_folder, model + ".pkl")
    print(f"Loading {model_id}")
    with open(file_path, 'rb') as f:
        model_dict = pickle.load(f)
    cols = {"model": model_id}
    cols.update(extract_answers(model_dict))
    rows.append(cols)

math_df = pd.DataFrame(rows)
math_df

coldata = {}
for file in files:
    file_path = os.path.join(import_dir, file)
    name = file[:-4]
    # print(f"Loading {name}...")
    try:
        with open(file_path, 'rb') as f:
            model_dict = pickle.load(f)
    except:
        # print("rm " + file, end=";")
        print("scp A100-7:/home/paperspace/pythia/output/" + file + " pythia_output/")
        print("scp A100-8:/home/paperspace/pythia/output/" + file + " pythia_output/")
        continue
    columns = []
    values = []
    for task in sorted(model_dict):
        values_dict = model_dict[task]
        if task == "dm_math_categories":
            continue
        losses = np.array([v.astype(float).sum() for v in values_dict['loss']])
        new_cols = [f"{task} {_id:03d}" for _id in values_dict['context_id']]
        columns.extend(new_cols[:take])
        values.extend(losses[:take])

    if old_cols is not None:
        assert columns == old_cols
    old_cols = columns
    coldata[name] = values
